# Tutorial: Fine-Tuning a Genomic Foundation Model for Translation Efficiency Prediction

Welcome to this educational tutorial on using OmniGenBench to tackle a key challenge in genomics: **Translation Efficiency (TE) Prediction**. Our goal is to fine-tune a powerful Genomic Foundation Model (GFM) to predict whether a given mRNA sequence from rice will be translated with high or low efficiency.

## The "What" and "Why": Translation Efficiency

**Translation Efficiency** is a critical measure in biology that quantifies how effectively an mRNA molecule is translated into a protein. It's a complex process governed by intricate sequence patterns within the mRNA itself. Understanding these patterns allows scientists to:
- **Design synthetic genes** with desired protein expression levels.
- **Uncover regulatory mechanisms** that control gene expression.
- **Improve crop yields** by optimizing protein production in plants.

Traditionally, predicting TE from sequence alone has been a difficult task. This is where modern AI, specifically Large Language Models, comes in.

## The Power of Genomic Foundation Models (GFMs)

Just as models like GPT have learned the patterns of human language, **Genomic Foundation Models (GFMs)** are trained on massive datasets of DNA and RNA sequences to learn the "language of the genome." By pre-training on billions of base pairs, these models develop a deep, intrinsic understanding of genomic syntax and grammar.

This pre-training makes them incredibly powerful for downstream tasks like ours. Instead of training a model from scratch, we can take a GFM and simply **fine-tune** it on our specific TE dataset. This process is faster, more data-efficient, and often leads to state-of-the-art performance. A notable example of a GFM in this domain is **PlantRNA-FM**, which demonstrated the power of these models for functional RNA analysis in plants.

## What You'll Learn in This Tutorial

We will guide you through a complete, hands-on pipeline for TE prediction using OmniGenBench. This tutorial is broken down into four clear steps:

1.  **Data Preparation**: We'll start by loading our specialized rice TE dataset and preparing it for the model.
2.  **Model Initialization**: Next, we'll load a pre-trained GFM and its tokenizer from the OmniGenBench model hub.
3.  **Model Training**: We will then fine-tune the GFM on our TE data, teaching it to distinguish between high and low-efficiency sequences.
4.  **Inference and Evaluation**: Finally, you'll learn how to use your new, specialized model to make predictions on novel sequences and evaluate its performance.

Let's get started!

## Step 1: Environment Setup

Before we dive in, we need to set up our environment by installing the necessary Python packages. This ensures we have all the tools we need for data processing, model training, and visualization.

If you have already installed these packages, you can skip the next cell.

In [ ]:
# Install the required packages
!pip install torch transformers pandas autocuda multimolecule biopython scipy scikit-learn tqdm dill findfile requests omnigenbench -U


With our environment ready, let's import the libraries we'll be using throughout the tutorial.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch, requests
import os
import zipfile
import findfile
from autocuda import auto_cuda
from omnigenbench import (
    ClassificationMetric,
    OmniTokenizer,
    OmniModelForSequenceClassification,
    OmniDatasetForSequenceClassification,
    Trainer,
    ModelHub,
)

warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

## Step 2: Data Preparation

A successful machine learning model starts with good data. In this step, we will:
1.  **Download and Unpack** our rice translation efficiency dataset.
2.  **Define a Custom Dataset Class** to process our sequences and labels.
3.  **Set up our Configuration**, including file paths and hyperparameters.

### 2.1 Download the Dataset

First, let's write a helper function to download and extract the dataset from the web. This function will check if the data already exists locally and, if not, fetch it from a public repository.

In [ ]:
def download_te_dataset(local_dir):
    """
    Downloads and extracts the Translation Efficiency dataset.
    """
    if not findfile.find_cwd_dir(local_dir, disable_alert=True):
        os.makedirs(local_dir, exist_ok=True)
    
    url_to_download = "https://huggingface.co/datasets/yangheng/translation_efficiency_prediction/resolve/main/translation_efficiency_prediction.zip"
    zip_path = os.path.join(local_dir, "te_rice_dataset.zip")

    if not os.path.exists(zip_path):
        print(f"Downloading te_rice_dataset.zip from {url_to_download}...")
        response = requests.get(url_to_download, stream=True)
        response.raise_for_status()

        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded {zip_path}")

    # Unzip the dataset
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(local_dir)
        print(f"Extracted te_rice_dataset.zip into {local_dir}")
        os.remove(zip_path)
    else:
        print("Dataset already downloaded and extracted.")

# --- Dataset Configuration ---
LOCAL_PATH = "te_rice_dataset" 
download_te_dataset(LOCAL_PATH)

# --- Define File Paths ---
TRAIN_FILE = findfile.find_cwd_file("train.json")
VALID_FILE = findfile.find_cwd_file("valid.json")
TEST_FILE = findfile.find_cwd_file("test.json")

print(f"Train file: {TRAIN_FILE}")
print(f"Validation file: {VALID_FILE}")
print(f"Test file: {TEST_FILE}")

### 2.2 Define a Custom Dataset Class

OmniGenBench provides a flexible `OmniDataset` class that we can extend to handle our specific data format. Here, we create `TEClassificationDataset` to:
-   Take a `sequence` and a `label` as input.
-   Use the model's **tokenizer** to convert the sequence string into numerical tokens.
-   Format the labels into the tensor format required for training.

This class acts as a bridge between our raw data files and the model.

In [ ]:
class TEClassificationDataset(OmniDatasetForSequenceClassification):
    def __init__(self, data_source, tokenizer, max_length, **kwargs):
        super().__init__(data_source, tokenizer, max_length, **kwargs)

    def prepare_input(self, instance, **kwargs):
        sequence, labels = instance["sequence"], instance["label"]

        # Tokenize the sequence
        tokenized_inputs = self.tokenizer(
            sequence,
            padding=True,
            truncation=True,

            max_length=self.max_length,
            return_tensors="pt",
            **kwargs
        )
        
        # Remove the extra batch dimension
        for col in tokenized_inputs:
            tokenized_inputs[col] = tokenized_inputs[col].squeeze(0)

        # Process labels
        if labels is not None:
            label_id = self.label2id.get(str(labels), -100)
            tokenized_inputs["labels"] = torch.tensor(label_id, dtype=torch.long)

        return tokenized_inputs


### 2.3 Configure Hyperparameters

Finally, let's define the key parameters for our experiment. This includes choosing a model, setting training hyperparameters, and defining the label mapping.

**Model Selection:** We'll start with a smaller, efficient GFM, `OmniGenome-52M`. OmniGenBench supports a variety of models, and you can easily swap this out to experiment with others.

**Hyperparameters:** These settings control the training process.
-   `EPOCHS`: The number of times we'll iterate over the entire training dataset.
-   `LEARNING_RATE`: Controls how much the model's weights are adjusted during training.
-   `BATCH_SIZE`: The number of sequences processed at once.
-   `MAX_LENGTH`: The maximum sequence length the model will handle.

In [ ]:
# --- Model Selection ---
MODEL_NAME = 'yangheng/OmniGenome-52M'

# --- Training Hyperparameters ---
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 4
MAX_LENGTH = 512
SEED = 42

# --- Label Mapping ---
# Our task is binary: 1 for High-TE, 0 for Low-TE.
LABEL2ID = {"0": 0, "1": 1}

print(f"Configuration set:")
print(f"  - Model: {MODEL_NAME}")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch Size: {BATCH_SIZE}")

## Step 3: Model Initialization

With our data prepared, it's time to bring in the star of the show: the Genomic Foundation Model. In this step, we will:
1.  **Load the Tokenizer**: This converts our text-based sequences into a numerical format the model understands.
2.  **Load the Model**: We'll instantiate a sequence classification model built upon our chosen GFM.

OmniGenBench makes this incredibly simple. We use `OmniTokenizer.from_pretrained` and `OmniModelForSequenceClassification` to load both components with just a few lines of code.

In [ ]:
# 1. Load Tokenizer
tokenizer = OmniTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# 2. Load Model for Sequence Classification
# We pass the model name, the tokenizer, and our label mapping.
model = OmniModelForSequenceClassification(
    MODEL_NAME,
    tokenizer=tokenizer,
    label2id=LABEL2ID,
    trust_remote_code=True,
)

print(f"Model '{MODEL_NAME}' and its tokenizer have been loaded successfully.")

## Step 4: Model Training

Now for the most exciting part: fine-tuning! Here, we'll take our pre-trained GFM and train it on our specific TE dataset. This process will adapt the model's general genomic knowledge to the specific task of TE prediction.

Our training pipeline involves:
1.  **Creating Dataset Instances**: We'll use our `TEClassificationDataset` class to create training, validation, and test sets.
2.  **Setting up DataLoaders**: These will efficiently feed batches of data to the model during training.
3.  **Defining Metrics and Optimizer**: We'll specify how to measure performance (F1 score) and how to update the model's weights (AdamW optimizer).
4.  **Using the `Trainer`**: OmniGenBench's `Trainer` class handles the entire training loop, including evaluation and saving the best model.

In [ ]:
# 1. Create Dataset instances
train_set = TEClassificationDataset(data_source=TRAIN_FILE, tokenizer=tokenizer, label2id=LABEL2ID, max_length=MAX_LENGTH)
valid_set = TEClassificationDataset(data_source=VALID_FILE, tokenizer=tokenizer, label2id=LABEL2ID, max_length=MAX_LENGTH)
test_set = TEClassificationDataset(data_source=TEST_FILE, tokenizer=tokenizer, label2id=LABEL2ID, max_length=MAX_LENGTH)

# 2. Create DataLoaders
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE)
print("Datasets and DataLoaders are ready.")

# 3. Define Metrics and Optimizer
# We'll use the F1 score to evaluate our classification model.
compute_metrics = [ClassificationMetric(ignore_y=-100, average="macro").f1_score]
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# 4. Initialize the Trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    eval_loader=valid_loader,
    test_loader=test_loader,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    optimizer=optimizer,
    compute_metrics=compute_metrics,
    seeds=SEED,
)

# Let's start training!
print("Starting the fine-tuning process...")
metrics = trainer.train()
trainer.save_model("finetuned_te_model")
print("Training complete! The fine-tuned model has been saved.")

In [ ]:
# Display final performance metrics
if metrics.get('valid'):
    print("\nValidation Set Performance (last epoch):")
    for key, value in metrics['valid'][-1].items():
        print(f"  - {key}: {value:.4f}")

if metrics.get('test'):
    print("\nTest Set Performance (best checkpoint):")
    for key, value in metrics['test'][-1].items():
        print(f"  - {key}: {value:.4f}")

# Visualize the training progress
valid_key = 'valid' if 'valid' in metrics else ('eval' if 'eval' in metrics else None)
if valid_key:
    valid_df = pd.DataFrame(metrics[valid_key])
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    sns.lineplot(data=valid_df, x=valid_df.index + 1, y='f1_score', ax=ax, marker='o', label='Validation F1 Score')
    
    ax.set_title('Validation F1 Score Across Epochs', fontsize=16)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('F1 Score (Macro)', fontsize=12)
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No validation metrics found to plot.")

## Step 5: Inference with the Fine-Tuned Model

Now that we have a fine-tuned model, how do we use it to make predictions on new, unseen sequences? This is the **inference** step.

OmniGenBench's `ModelHub` makes this incredibly easy. We can load our trained model with a single line of code and then call the `.inference()` method on any sequence we want to analyze.

Let's test it with two example sequences: one known to have high TE and one with low TE.

## Conclusion and Next Steps

Congratulations! You have successfully fine-tuned a Genomic Foundation Model for Translation Efficiency prediction.

In this tutorial, you have learned how to:
- **Structure a genomics ML project** with clear steps for data preparation, modeling, training, and inference.
- **Leverage OmniGenBench** to easily load pre-trained models and tokenizers.
- **Implement a custom dataset class** to process your specific data format.
- **Use the `Trainer` API** to handle the complexities of the training and evaluation loop.
- **Make predictions** on new sequences with your fine-tuned model.

From here, you can explore:
- **Experimenting with different GFMs** by changing the `MODEL_NAME`.
- **Tuning hyperparameters** like the learning rate or batch size to improve performance.
- **Applying this pipeline** to your own sequence classification tasks.

### References
- PlantRNA-FM: "An interpretable RNA foundation model for exploring functional RNA motifs in plants" (Nature Machine Intelligence, 2024).

In [ ]:
# Load the fine-tuned model from the hub
# Note: Replace with the correct path if you saved it locally
# For this example, we load a pre-finetuned model from the hub
inference_model = ModelHub.load("yangheng/ogb_te_finetuned").to(auto_cuda())

# Example sequences
high_te_sequence = "AAACCAACAAAATGCAGTAGAAGTACTCTCGAGCTATAGTCGCGACGTGCTGCCCCGCAGGAGTACAGTAGTAGTACAACGTAAGCGGGAGCAACAGACTCCCCCCCTGCAACCCACTGTGCCTGTGCCCTCGACGCGTCTCCGTCGCTTTGGCAAATGTCACGTACATATTACCGTCTCAGGCTCTCAGCCATGCTCCCTACCACCCCTGCAGCGAAGCAAAAGCCACGCACGCGGCGCCTGACATGTAACAGGACTAGACCATCTTGTTCATTTCCCGCACCCCCTCCTCTCCTCTTCCTCCATCTGCCTCTTTAAAACAGTAAAAATAACCGTGCATCCCCTGGGCAAAATCTCTCCCATACATACACTACAGCGGCGAACCTTTCCTTATTCTCGCAACGCCTCGGTAACGGGCAGCGCCTGCTCCGCGCCGCGGTTGCGAGTTCGGGAAGGCGGCCGGAGTCGCGGGGAGGAGAGGGAGGATTCGATCGGCCAGA"
low_te_sequence = "TGGAGATGGGCAGATGGCACACAAAACATGAATAGAAAACCCAAAAGGAAGGATGAAAAAAACACACACACACACACACACAAAACACAGAGAGAGAGAGAGAGAGAGAGCGAGAAAAGAAAAGAAAAAACCAATTCTTTTGGTCTCTTCCCTCTCCGTTTGTCGTGTCGAAGCCTTTGCCCCCACCACCTCCTCCTCTCCTCTCCCTTCCTCCCCTCCTCCCCATCTCGCTCTCCTCCCTCCTCTCTCCTCTCCTCGTCTCCTCTTCCTCTCCATTCCATTGGCCATTCCATTCCATTCCACCCCCCATGAAACCCCAAACCCTCGTCGGCCTCGCCGCGCTCGCGTAGCGCACCCGCCCTTCTCCTCTCGCCGGTGGTCCGCCGCCAGCCTCCCCCCACCCGATCCCGCCGCCCCCCCCGCCTTCACCCCGCCCACGCGGACGCATCCGATCCCGCCGCATCGCCGCGCGGGGGGGGGGGGGGGGGGGGGGGAGGGCACG"

# Run inference
high_te_output = inference_model.inference(high_te_sequence)
low_te_output = inference_model.inference(low_te_sequence)

print(f"Prediction for High-TE sequence: {high_te_output['predictions']} (Label: {high_te_output['label']})")
print(f"Prediction for Low-TE sequence: {low_te_output['predictions']} (Label: {low_te_output['label']})")
